In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-30

@author:       Oscar Trevizo
@institution:  Harvard Extension School
@credential:   Graduate Certificate in Data Science (2023)
@context:      Independent project — applying graduate-level concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Agentic Cross-Metric Tool — Experiment 3
==========================================

Context:
    Experiment 1 showed that LLM agents diverge on cross-metric reasoning questions.
    Experiment 2 showed that the semantic layer is useful but not the correctness driver.
    The decisive failure in both experiments was the year filter — a probabilistic
    neural reasoning choice that the symbolic layer could not enforce.

Hypothesis:
    A governed cross-metric tool with year as a required parameter will
    eliminate the year-filter failure and produce consistent correct answers
    across all models.

Design (2x2 matrix completing Experiments 1 and 2):

    |                     | Semantic layer  | No semantic layer |
    |---------------------|-----------------|-------------------|
    | No cross-metric tool| Exp 2 Cond A    | Exp 2 Cond B      |
    | Cross-metric tool   | Exp 3 Cond A    | Exp 3 Cond B      |

    Experiment 3 fills the bottom row. Comparing columns isolates the
    semantic layer effect when the tool is governed. Comparing rows isolates
    the tool structure effect within each naming condition.

New tool: query_cross_metric(product_name, rank_metric, filter_metric,
                              filter_top_n, year)
    - Performs the cross-metric join in one governed call
    - year is REQUIRED in the schema — model cannot call without specifying it
    - Condition A: rank_metric and filter_metric are semantic business names
    - Condition B: rank_metric and filter_metric are raw column names

Question: "Which country in the top 20 GDP has the highest migration rate?"
Models:   GPT-4o-mini · Haiku · Sonnet

Precursors:
    machine_learning/agentic_model_reasoning_divergence.ipynb      (Exp 1)
    machine_learning/agentic_semantic_layer_necessity.ipynb        (Exp 2)

Revision History:
    2026-06-30  Original development
                - query_cross_metric() with year as required parameter
                - Two conditions: semantic business names vs raw column names
                - Completes the 2x2 experimental matrix
"""

## Hypothesis

> **A governed cross-metric tool with `year` as a required parameter will
> eliminate the year-filter failure and produce consistent correct answers
> across all models.**

### The structural gap (from Experiments 1 and 2)

No tool in prior experiments returned *"migration rate filtered to the top-20 GDP countries."*
Models had to:
1. Query GDP top 20 (all years, no filter)
2. Query migration rates (all countries, no filter)
3. Mentally intersect the two result sets
4. Decide whether to apply a year filter

Step 4 was the coin flip — Haiku always applied it, Sonnet probabilistically,
GPT-4o-mini never. The semantic layer could not enforce it.

### The intervention

`query_cross_metric()` performs all four steps in one governed call.
`year` is marked **required** in the JSON schema — the API will reject the call
if the model does not supply it. This removes the probabilistic failure mode entirely.

### What this isolates

| | Semantic layer | No semantic layer |
|---|---|---|
| **No cross-metric tool** | Exp 2 Cond A (4/9 correct) | Exp 2 Cond B (5/9 correct) |
| **Cross-metric tool** | **Exp 3 Cond A** | **Exp 3 Cond B** |

If Exp 3 is correct across all models in both conditions → the structural gap was the only issue.
If Exp 3 Cond A correct but Cond B wrong → semantic layer IS necessary with a better tool.
If Exp 3 still fails → the failure is deeper than tool structure or naming.

In [ ]:
import io
import json
import os
import sys
from datetime import datetime, timezone

import anthropic
import numpy as np
import pandas as pd
from openai import OpenAI

sys.path.insert(0, '../python_vignettes/data_products/')
from data_product_lib import (
    DataProductMetadata,
    SemanticLayer,
    LineageTracker,
    DataProduct,
)

anthropic_client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
openai_client    = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

print(f"Anthropic SDK : {anthropic.__version__}")
import openai as _oai
print(f"OpenAI SDK    : {_oai.__version__}")

## Experimental Design

| Element | Value |
|---|---|
| **Question** | *"Which country in the top 20 GDP has the highest migration rate?"* |
| **Models** | GPT-4o-mini · Haiku · Sonnet |
| **Condition A** | `query_cross_metric(rank_metric=..., filter_metric=..., year=...)` — semantic names, year required |
| **Condition B** | `query_cross_metric_raw(rank_column=..., filter_column=..., year=...)` — raw names, year required |
| **Key intervention** | `year` is a required schema parameter — model cannot omit it |
| **Controlled** | Data, pipeline, merge, question, year requirement |
| **Variable** | Semantic business names vs raw column names |
| **Prior results** | Exp 2 Cond A: 4/9 correct · Exp 2 Cond B: 5/9 correct |

### What the cross-metric tool does differently

```
Prior experiments (two separate query calls):
  query_product(business_name='gdp', top_n=20)            → top 20 by GDP, all years
  query_product(business_name='net_migration_rate', top_n=N) → top N by migration, all years
  [model reasons across two disconnected result sets]

Experiment 3 (one governed call):
  query_cross_metric(
      filter_metric='gdp',           # which metric defines the filter set
      filter_top_n=20,               # how many countries to keep
      rank_metric='net_migration_rate', # which metric to rank by within that set
      year=2023                      # REQUIRED — model must specify
  )  → top 20 GDP countries in 2023, ranked by migration rate in 2023
```

In [ ]:
UN_FILE            = '../data/WPP2024_GEN_F01_DEMOGRAPHIC_INDICATORS_COMPACT_20260327.xlsx'
WB_GDP_FILE        = '../data/WB_GDP.xls'
WB_GDP_GROWTH_FILE = '../data/WB_GDP_growth.xls'


def load_un_wpp(filepath, year_start=1961, year_end=2023):
    df = pd.read_excel(
        filepath, sheet_name='Estimates', skiprows=16, thousands=' ',
        usecols=[
            'Region, subregion, country or area *', 'ISO3 Alpha-code',
            'ISO2 Alpha-code', 'Type', 'Year',
            'Total Population, as of 1 July (thousands)',
            'Median Age, as of 1 July (years)',
            'Population Growth Rate (percentage)',
            'Total Fertility Rate (live births per woman)',
            'Life Expectancy at Birth, both sexes (years)',
            'Net Number of Migrants (thousands)',
            'Net Migration Rate (per 1,000 population)',
        ]
    )
    df = df.rename(columns={
        'Region, subregion, country or area *'         : 'Location',
        'ISO3 Alpha-code'                              : 'ISO3',
        'ISO2 Alpha-code'                              : 'ISO2',
        'Type'                                         : 'LocType',
        'Total Population, as of 1 July (thousands)'  : 'Population_Ks',
        'Median Age, as of 1 July (years)'             : 'MedAge',
        'Population Growth Rate (percentage)'          : 'PopulationGrowthRate',
        'Total Fertility Rate (live births per woman)' : 'FertilityRate_births_per_woman',
        'Life Expectancy at Birth, both sexes (years)' : 'LifeExpectancy',
        'Net Number of Migrants (thousands)'           : 'NetMigrants_Ks',
        'Net Migration Rate (per 1,000 population)'    : 'NetMigrationRate_per_Kpop',
    })
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(np.int64)
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    df.loc[df['LocType'] == 'Country/Area', 'LocType'] = 'Country'
    df['ReceivesMigrants']    = (df['NetMigrants_Ks'] > 0).astype(int)
    df['ImmigrantsEmigrants'] = df['NetMigrants_Ks'].apply(
        lambda x: 'Immigrants' if x > 0 else 'Emigrants')
    return df.reset_index(drop=True)


def filter_countries(df):
    return df[df['LocType'] == 'Country'].reset_index(drop=True)


def clean_types(df):
    for col in ['Population_Ks', 'MedAge', 'PopulationGrowthRate',
                'FertilityRate_births_per_woman', 'LifeExpectancy',
                'NetMigrants_Ks', 'NetMigrationRate_per_Kpop']:
        df = df.copy()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_wb_gdp(gdp_file, gdp_growth_file, year_start=1961, year_end=2023):
    def _load_one(filepath, indicator):
        df = pd.read_excel(filepath, skiprows=3, sheet_name='Data')
        df = df.rename(columns={'Country Code': 'ISO3'})
        df = df.drop(columns=['Country Name', 'Indicator Name', 'Indicator Code'])
        df = df.drop(columns=[c for c in df.columns if str(c) == '2025'], errors='ignore')
        df = df.set_index('ISO3').stack().reset_index()
        df.columns = ['ISO3', 'Year', indicator]
        df['Year']    = df['Year'].astype(np.int64)
        df[indicator] = pd.to_numeric(df[indicator], errors='coerce')
        return df
    df = pd.merge(_load_one(gdp_file, 'GDP_USD'),
                  _load_one(gdp_growth_file, 'GDP_growth_pct'),
                  on=['ISO3', 'Year'])
    df = df[(df['Year'] >= year_start) & (df['Year'] <= year_end)]
    return df.reset_index(drop=True)


def _merge_data_products(dp1, dp2, on):
    collisions = set(dp1.semantic_layer.to_dict()) & set(dp2.semantic_layer.to_dict())
    if collisions:
        raise ValueError(f'Semantic name collision(s): {sorted(collisions)}.')
    merged_df = pd.merge(dp1.data, dp2.data, on=on, how='inner')
    merged_sl = SemanticLayer()
    for name, e in dp1.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    for name, e in dp2.semantic_layer.to_dict().items():
        merged_sl.register(name, e['column'], e['unit'],
                           e['description'], e['source_system'])
    merged_lin = LineageTracker()
    for s in dp1.lineage.to_list():
        merged_lin.log(f"dp1/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    for s in dp2.lineage.to_list():
        merged_lin.log(f"dp2/{s['step']}", s['operation'],
                       s['input_shape'], s['output_shape'], s['notes'])
    merged_lin.log('merge', 'inner_join', dp1.data.shape, merged_df.shape,
                   f'keys: {"+".join(on)}')
    return merged_df, merged_sl, merged_lin


print("Pipeline functions ready.")

In [ ]:
PRODUCT_REGISTRY: dict = {}

SOURCE_CONFIGS = {
    "UN_WPP":         {"description": "UN World Population Prospects 2024"},
    "WORLD_BANK_GDP": {"description": "World Bank WDI — GDP indicators"},
}


def list_available_sources() -> dict:
    return {"available_sources": SOURCE_CONFIGS,
            "loaded": list(PRODUCT_REGISTRY.keys())}


def load_source(source_name: str,
                year_start: int = 1961, year_end: int = 2023) -> dict:
    if source_name in PRODUCT_REGISTRY:
        dp = PRODUCT_REGISTRY[source_name]
        return {"status": "already_loaded", "product_name": source_name,
                "shape": list(dp.data.shape)}
    if source_name == "UN_WPP":
        lin = LineageTracker()
        raw  = load_un_wpp(UN_FILE, year_start, year_end)
        lin.log("1-load",   "load_un_wpp",        (0, 0),    raw.shape,  f"{year_start}-{year_end}")
        ctry = filter_countries(raw)
        lin.log("2-filter", "filter_countries",    raw.shape, ctry.shape, "LocType==Country")
        df   = clean_types(ctry)
        lin.log("3-clean",  "clean_types",          ctry.shape, df.shape, "to_numeric")
        sl = SemanticLayer()
        sl.register("net_migration_rate", "NetMigrationRate_per_Kpop", "per 1,000 population",
                    "Net migrants per 1,000 residents", source_system="UN WPP 2024")
        sl.register("net_migrants", "NetMigrants_Ks", "thousands",
                    "Net number of migrants", source_system="UN WPP 2024")
        sl.register("population", "Population_Ks", "thousands",
                    "Total population, as of 1 July", source_system="UN WPP 2024")
        sl.register("fertility_rate", "FertilityRate_births_per_woman", "births per woman",
                    "Total fertility rate", source_system="UN WPP 2024")
        sl.register("life_expectancy", "LifeExpectancy", "years",
                    "Life expectancy at birth", source_system="UN WPP 2024")
        meta = DataProductMetadata(
            name="UN WPP Demographics", description="Country-year panel, UN WPP 2024",
            domain="Demographics", owner="Experiment",
            source="UN World Population Prospects 2024",
            source_url="https://population.un.org/wpp/",
            license="CC BY 3.0 IGO", version="1.0",
            refresh_frequency="Every 2 years",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    elif source_name == "WORLD_BANK_GDP":
        lin = LineageTracker()
        df  = load_wb_gdp(WB_GDP_FILE, WB_GDP_GROWTH_FILE, year_start, year_end)
        lin.log("1-load", "load_wb_gdp", (0, 0), df.shape, f"{year_start}-{year_end}")
        sl = SemanticLayer()
        sl.register("gdp", "GDP_USD", "constant USD",
                    "GDP (NY.GDP.MKTP.KD)", source_system="World Bank WDI")
        sl.register("gdp_growth", "GDP_growth_pct", "% per year",
                    "GDP growth (NY.GDP.MKTP.KD.ZG)", source_system="World Bank WDI")
        meta = DataProductMetadata(
            name="World Bank GDP", description="Country-year panel, World Bank WDI",
            domain="Economics", owner="Experiment",
            source="World Bank World Development Indicators",
            source_url="https://databank.worldbank.org/",
            license="CC BY 4.0", version="1.0",
            refresh_frequency="Annual",
            created_at=datetime.now(timezone.utc).isoformat())
        dp = DataProduct(df, meta, sl, lin)
    else:
        return {"error": f"Unknown source '{source_name}'."}
    PRODUCT_REGISTRY[source_name] = dp
    return {"status": "loaded", "product_name": source_name,
            "shape": list(dp.data.shape),
            "semantic_names": list(dp.semantic_layer.to_dict().keys())}


def merge_sources(source1_name, source2_name, product_name="MERGED"):
    missing = [n for n in (source1_name, source2_name) if n not in PRODUCT_REGISTRY]
    if missing:
        return {"error": f"Not in registry: {missing}"}
    dp1, dp2 = PRODUCT_REGISTRY[source1_name], PRODUCT_REGISTRY[source2_name]
    try:
        mdf, msl, mlin = _merge_data_products(dp1, dp2, on=["ISO3", "Year"])
    except ValueError as exc:
        return {"error": str(exc)}
    meta = DataProductMetadata(
        name=product_name, description=f"Merged: {source1_name}+{source2_name}",
        domain="Multi-domain", owner="Experiment",
        source=f"{dp1.metadata.source}+{dp2.metadata.source}",
        source_url="", license="", version="1.0",
        refresh_frequency="On demand",
        created_at=datetime.now(timezone.utc).isoformat())
    dp = DataProduct(mdf, meta, msl, mlin)
    PRODUCT_REGISTRY[product_name] = dp
    return {"status": "merged", "product_name": product_name,
            "shape": list(dp.data.shape),
            "semantic_names": sorted(dp.semantic_layer.to_dict().keys())}


print("Shared tools ready (list_available_sources, load_source, merge_sources).")

---
## Condition A — Cross-Metric Tool + Semantic Layer

`query_cross_metric()` takes semantic business names and performs the
full cross-metric join in one governed call. `year` is required in the schema.

In [ ]:
def query_cross_metric(product_name, filter_metric, filter_top_n, rank_metric, year):
    # Governed cross-metric join. year is required — enforced by schema.
    # 1. Filter to top filter_top_n countries by filter_metric in year.
    # 2. Among those, rank by rank_metric in year.
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    sl = dp.semantic_layer.to_dict()
    for name in (filter_metric, rank_metric):
        if name not in sl:
            return {"error": f"'{name}' not in semantic layer",
                    "available": sorted(sl.keys())}
    filter_col = sl[filter_metric]['column']
    rank_col   = sl[rank_metric]['column']
    df_year = dp.data[dp.data['Year'] == year].copy()
    if df_year.empty:
        return {"error": f"No data for year {year}"}
    top_iso3 = df_year.nlargest(filter_top_n, filter_col)['ISO3'].tolist()
    result = (df_year[df_year['ISO3'].isin(top_iso3)]
              .nlargest(filter_top_n, rank_col)
              .reset_index(drop=True))
    cols = [c for c in ['Location', 'ISO3', 'Year', filter_col, rank_col]
            if c in result.columns]
    return {
        "filter_metric": filter_metric, "filter_col": filter_col,
        "filter_top_n": filter_top_n,
        "rank_metric": rank_metric, "rank_col": rank_col,
        "year": year, "result_count": len(result),
        "results": result[cols].to_dict(orient="records"),
    }


TOOL_FUNCTIONS_A = {
    "list_available_sources": list_available_sources,
    "load_source":            load_source,
    "merge_sources":          merge_sources,
    "query_cross_metric":     query_cross_metric,
}

TOOLS_ANTHROPIC_A = [
    {"name": "list_available_sources",
     "description": "Return available data sources and which are loaded.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "load_source",
     "description": "Load a source ('UN_WPP' or 'WORLD_BANK_GDP') and register it.",
     "input_schema": {"type": "object",
       "properties": {"source_name": {"type": "string"},
                      "year_start":  {"type": "integer"},
                      "year_end":    {"type": "integer"}},
       "required": ["source_name"]}},
    {"name": "merge_sources",
     "description": "Inner join two DataProducts on ISO3+Year.",
     "input_schema": {"type": "object",
       "properties": {"source1_name": {"type": "string"},
                      "source2_name": {"type": "string"},
                      "product_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "query_cross_metric",
     "description": (
         "Filter a merged product to the top filter_top_n countries by filter_metric "
         "in the specified year, then rank those countries by rank_metric in the same year. "
         "Semantic business names: 'gdp', 'net_migration_rate', 'population', etc. "
         "year is required."),
     "input_schema": {"type": "object",
       "properties": {
           "product_name":  {"type": "string"},
           "filter_metric": {"type": "string",
                             "description": "Semantic name of the metric used to define the filter set (e.g. 'gdp')"},
           "filter_top_n":  {"type": "integer",
                             "description": "Number of top countries to keep by filter_metric"},
           "rank_metric":   {"type": "string",
                             "description": "Semantic name of the metric to rank by within the filtered set (e.g. 'net_migration_rate')"},
           "year":          {"type": "integer",
                             "description": "Year to filter data to. Required."},
       },
       "required": ["product_name", "filter_metric", "filter_top_n", "rank_metric", "year"]}},
]

TOOLS_OPENAI_A = [
    {"type": "function", "function": {
        "name": t["name"], "description": t["description"],
        "parameters": t["input_schema"]}}
    for t in TOOLS_ANTHROPIC_A
]

SYSTEM_PROMPT_A = (
    "You are a data analyst agent. Use the tools to answer the user's question.\n"
    "1. Call list_available_sources() to see what data is available.\n"
    "2. Call load_source() for each source you need.\n"
    "3. Call merge_sources() to combine them.\n"
    "4. Call query_cross_metric() to answer the question in one governed call.\n"
    "   Specify filter_metric, filter_top_n, rank_metric, and year (required).\n"
    "   Semantic names available: gdp, net_migration_rate, population, gdp_growth, etc.\n"
    "Be concise. Report the country name and value in your final answer."
)

print(f"Condition A: {len(TOOL_FUNCTIONS_A)} tools "
      f"(cross-metric + semantic layer, year required).")

---
## Condition B — Cross-Metric Tool + Raw Column Names

`query_cross_metric_raw()` takes raw technical column names.
`year` is still required. `list_columns()` is available for discovery.

In [ ]:
def list_columns(product_name):
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    return {"product_name": product_name,
            "columns": list(dp.data.columns),
            "shape": list(dp.data.shape)}


def query_cross_metric_raw(product_name, filter_column, filter_top_n,
                            rank_column, year):
    # Same logic as query_cross_metric but with raw column names.
    # year is required — enforced by schema.
    if product_name not in PRODUCT_REGISTRY:
        return {"error": f"'{product_name}' not in registry"}
    dp = PRODUCT_REGISTRY[product_name]
    for col in (filter_column, rank_column):
        if col not in dp.data.columns:
            return {"error": f"'{col}' not a column in '{product_name}'",
                    "available_columns": list(dp.data.columns)}
    df_year = dp.data[dp.data['Year'] == year].copy()
    if df_year.empty:
        return {"error": f"No data for year {year}"}
    top_iso3 = df_year.nlargest(filter_top_n, filter_column)['ISO3'].tolist()
    result = (df_year[df_year['ISO3'].isin(top_iso3)]
              .nlargest(filter_top_n, rank_column)
              .reset_index(drop=True))
    cols = [c for c in ['Location', 'ISO3', 'Year', filter_column, rank_column]
            if c in result.columns]
    return {
        "filter_column": filter_column, "filter_top_n": filter_top_n,
        "rank_column": rank_column, "year": year,
        "result_count": len(result),
        "results": result[cols].to_dict(orient="records"),
    }


TOOL_FUNCTIONS_B = {
    "list_available_sources":  list_available_sources,
    "load_source":             load_source,
    "merge_sources":           merge_sources,
    "list_columns":            list_columns,
    "query_cross_metric_raw":  query_cross_metric_raw,
}

TOOLS_ANTHROPIC_B = [
    {"name": "list_available_sources",
     "description": "Return available data sources and which are loaded.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "load_source",
     "description": "Load a source ('UN_WPP' or 'WORLD_BANK_GDP') and register it.",
     "input_schema": {"type": "object",
       "properties": {"source_name": {"type": "string"},
                      "year_start":  {"type": "integer"},
                      "year_end":    {"type": "integer"}},
       "required": ["source_name"]}},
    {"name": "merge_sources",
     "description": "Inner join two DataProducts on ISO3+Year.",
     "input_schema": {"type": "object",
       "properties": {"source1_name": {"type": "string"},
                      "source2_name": {"type": "string"},
                      "product_name": {"type": "string"}},
       "required": ["source1_name", "source2_name"]}},
    {"name": "list_columns",
     "description": "List all raw column names in a registered product.",
     "input_schema": {"type": "object",
       "properties": {"product_name": {"type": "string"}},
       "required": ["product_name"]}},
    {"name": "query_cross_metric_raw",
     "description": (
         "Filter a merged product to the top filter_top_n countries by filter_column "
         "in the specified year, then rank those countries by rank_column in the same year. "
         "Use list_columns() first to discover available column names. "
         "year is required."),
     "input_schema": {"type": "object",
       "properties": {
           "product_name":  {"type": "string"},
           "filter_column": {"type": "string",
                             "description": "Raw column name used to define the filter set (e.g. 'GDP_USD')"},
           "filter_top_n":  {"type": "integer",
                             "description": "Number of top countries to keep by filter_column"},
           "rank_column":   {"type": "string",
                             "description": "Raw column name to rank by within the filtered set (e.g. 'NetMigrationRate_per_Kpop')"},
           "year":          {"type": "integer",
                             "description": "Year to filter data to. Required."},
       },
       "required": ["product_name", "filter_column", "filter_top_n",
                    "rank_column", "year"]}},
]

TOOLS_OPENAI_B = [
    {"type": "function", "function": {
        "name": t["name"], "description": t["description"],
        "parameters": t["input_schema"]}}
    for t in TOOLS_ANTHROPIC_B
]

SYSTEM_PROMPT_B = (
    "You are a data analyst agent. Use the tools to answer the user's question.\n"
    "1. Call list_available_sources() to see what data is available.\n"
    "2. Call load_source() for each source you need.\n"
    "3. Call merge_sources() to combine them.\n"
    "4. Call list_columns() to discover available column names.\n"
    "5. Call query_cross_metric_raw() to answer the question in one governed call.\n"
    "   Specify filter_column, filter_top_n, rank_column, and year (required).\n"
    "Be concise. Report the country name and value in your final answer."
)

print(f"Condition B: {len(TOOL_FUNCTIONS_B)} tools "
      f"(cross-metric + raw column names, year required).")

In [ ]:
MAX_ITERATIONS = 20


def run_agent(question, provider, model, condition_label,
              tools_anthropic, tools_openai, tool_functions,
              system_prompt, verbose=True):
    PRODUCT_REGISTRY.clear()
    tool_calls_log = []

    if provider == "anthropic":
        messages = [{"role": "user", "content": question}]
    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ]

    for iteration in range(MAX_ITERATIONS):

        if provider == "anthropic":
            response = anthropic_client.messages.create(
                model=model, max_tokens=2048,
                system=system_prompt,
                tools=tools_anthropic,
                messages=messages)

            if response.stop_reason == "end_turn":
                text = next(
                    (b.text for b in response.content if hasattr(b, "text")), "")
                return {"answer": text, "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider,
                        "condition": condition_label}

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                results = []
                for block in response.content:
                    if block.type == "tool_use":
                        tool_calls_log.append({"name": block.name, "input": block.input})
                        if verbose:
                            print(f"  -> {block.name}({block.input})")
                        fn     = tool_functions.get(block.name)
                        result = fn(**block.input) if fn else {"error": "unknown tool"}
                        results.append({"type": "tool_result",
                                        "tool_use_id": block.id,
                                        "content": json.dumps(result)})
                messages.append({"role": "user", "content": results})

        elif provider == "openai":
            response = openai_client.chat.completions.create(
                model=model, max_tokens=2048,
                tools=tools_openai,
                messages=messages)

            choice = response.choices[0]

            if choice.finish_reason == "stop":
                return {"answer": choice.message.content,
                        "tool_calls": tool_calls_log,
                        "iterations": iteration + 1,
                        "model": model, "provider": provider,
                        "condition": condition_label}

            if choice.finish_reason == "tool_calls":
                messages.append(choice.message)
                for tc in choice.message.tool_calls:
                    args = json.loads(tc.function.arguments)
                    tool_calls_log.append({"name": tc.function.name, "input": args})
                    if verbose:
                        print(f"  -> {tc.function.name}({args})")
                    fn     = tool_functions.get(tc.function.name)
                    result = fn(**args) if fn else {"error": "unknown tool"}
                    messages.append({"role": "tool",
                                     "tool_call_id": tc.id,
                                     "content": json.dumps(result)})

    return {"answer": "Max iterations reached.",
            "tool_calls": tool_calls_log, "iterations": MAX_ITERATIONS,
            "model": model, "provider": provider, "condition": condition_label}


print("run_agent() ready.")

In [ ]:
QUESTION = (
    "Which country in the top 20 GDP has the highest migration rate?"
)

MODELS = [
    {"provider": "openai",    "model": "gpt-4o-mini",               "label": "OpenAI GPT-4o-mini"},
    {"provider": "anthropic", "model": "claude-haiku-4-5-20251001",  "label": "Anthropic Haiku"},
    {"provider": "anthropic", "model": "claude-sonnet-4-6",          "label": "Anthropic Sonnet"},
]

print(f"Question : {QUESTION}")
print(f"Models   : {[m['label'] for m in MODELS]}")

---
## Run — Condition A (Cross-Metric Tool + Semantic Layer)

In [ ]:
results_a = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"[A] {cfg['label']}")
    print(f"{'='*60}")
    result = run_agent(
        QUESTION, cfg["provider"], cfg["model"],
        condition_label="A",
        tools_anthropic=TOOLS_ANTHROPIC_A,
        tools_openai=TOOLS_OPENAI_A,
        tool_functions=TOOL_FUNCTIONS_A,
        system_prompt=SYSTEM_PROMPT_A,
        verbose=True)
    result["label"] = cfg["label"]
    results_a.append(result)
    print(f"\nAnswer:\n{result['answer']}")

---
## Run — Condition B (Cross-Metric Tool + Raw Column Names)

In [ ]:
results_b = []

for cfg in MODELS:
    print(f"\n{'='*60}")
    print(f"[B] {cfg['label']}")
    print(f"{'='*60}")
    result = run_agent(
        QUESTION, cfg["provider"], cfg["model"],
        condition_label="B",
        tools_anthropic=TOOLS_ANTHROPIC_B,
        tools_openai=TOOLS_OPENAI_B,
        tool_functions=TOOL_FUNCTIONS_B,
        system_prompt=SYSTEM_PROMPT_B,
        verbose=True)
    result["label"] = cfg["label"]
    results_b.append(result)
    print(f"\nAnswer:\n{result['answer']}")

---
## Results

In [ ]:
def summarise(results, condition_label):
    rows = []
    for r in results:
        correct = 'canada' in r['answer'].lower()
        cross_calls = [tc for tc in r['tool_calls']
                       if tc['name'] in ('query_cross_metric',
                                         'query_cross_metric_raw')]
        years_used = [tc['input'].get('year') for tc in cross_calls
                      if tc['input'].get('year')]
        rows.append({
            'Condition': condition_label,
            'Model':     r['label'],
            'Tool calls': len(r['tool_calls']),
            'Cross-metric calls': len(cross_calls),
            'Year(s) used': str(years_used) if years_used else 'none',
            'Canada correct?': 'YES' if correct else 'NO',
        })
    return rows

rows = summarise(results_a, 'A') + summarise(results_b, 'B')
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print("\n--- 2x2 Matrix (vs Experiment 2) ---")
matrix = {
    'Exp 2 No cross-metric, semantic (4/9)':    'Exp 2 No cross-metric, raw (5/9)',
    'Exp 3 Cross-metric, semantic (this run)':  'Exp 3 Cross-metric, raw (this run)',
}
for a, b in matrix.items():
    print(f"  {a}  |  {b}")

---
## Findings

Fill in after running the experiment.

### 2×2 matrix — Experiment 2 vs Experiment 3

| | Semantic layer | No semantic layer |
|---|---|---|
| **No cross-metric tool (Exp 2)** | 4/9 correct | 5/9 correct |
| **Cross-metric tool (Exp 3)** | *observed* | *observed* |

### What to look for

- Did making `year` required eliminate GPT-4o-mini's failures?
- Did all three models call `query_cross_metric()` with a year on the first try?
- Did any model try to work around the tool (call it multiple times, or fall back to old patterns)?
- Does the semantic layer matter now that the tool structure is governed?

### Hypothesis verdict

Fill in after observing results.

### Implication for the neurosymbolic hypothesis

If the cross-metric tool fixes correctness across all models and both conditions:
→ The structural gap was the root cause. Tool design > naming > model choice.

If Condition A correct but Condition B wrong:
→ The semantic layer IS necessary in combination with a governed tool.
→ Experiment 2's conclusion ("useful but not necessary") was conditional on tool quality.

If failures persist even with the governed tool:
→ The failure is deeper — model reasoning quality independent of tool design or naming.

---
## References

- **Data:** UN WPP 2024 (population.un.org/wpp) · World Bank WDI (databank.worldbank.org)
- **Anthropic Tool Use:** docs.anthropic.com/en/docs/tool-use
- **OpenAI Function Calling:** platform.openai.com/docs/guides/function-calling
- **Data products:** Dehghani (2022) *Data Mesh*, O'Reilly
- **Experiment 1:** `machine_learning/agentic_model_reasoning_divergence.ipynb`
- **Experiment 2:** `machine_learning/agentic_semantic_layer_necessity.ipynb`
- **Experiment 3:** `machine_learning/agentic_cross_metric_tool.ipynb` (this notebook)
- **Support module:** `../python_vignettes/data_products/data_product_lib.py`